# Retrieval evaluation with `epzero.ir`

Dense, keyword and hybrid retrieval on a BEIR dataset (SciFact: 5,183 abstracts, 300 test claims), scored
the way TREC and CLEF score submissions. The same code writes a run file you could submit.

```bash
pip install epzero sentence-transformers
```

In [1]:
import io
import os
import time
import urllib.request
import zipfile

import numpy as np
from sentence_transformers import SentenceTransformer

from epzero import ir

if not os.path.isdir("scifact"):
    url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/scifact.zip"
    zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall(".")
corpus, queries, qrels = ir.load_beir("scifact")
print(len(corpus), "documents,", len(queries), "queries with judgements")

5183 documents, 300 queries with judgements


## Embed and index

`Collection.build` keys the documents by their own ids, builds a vector index with `epzero.build` and a
BM25 index of the text.

In [2]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
doc_ids, texts = list(corpus), list(corpus.values())
query_ids, query_texts = list(queries), list(queries.values())
doc_vectors = model.encode(texts, batch_size=64, normalize_embeddings=True)
query_vectors = model.encode(query_texts, normalize_embeddings=True)

start = time.perf_counter()
collection = ir.Collection.build(doc_ids, doc_vectors, texts=texts, metric="cosine")
print(f"indexed in {time.perf_counter() - start:.2f} s")

indexed in 0.64 s


## Three runs

Keyword (BM25 with English stemming), dense, and hybrid: both, merged by reciprocal rank fusion.

In [3]:
runs = {
    "keyword": collection.retrieve(query_ids, texts=query_texts, k=100, mode="keyword"),
    "dense": collection.retrieve(query_ids, vectors=query_vectors, k=100),
    "hybrid": collection.retrieve(query_ids, vectors=query_vectors, texts=query_texts, k=100, mode="hybrid"),
}
for name, run in runs.items():
    scores = ir.evaluate(run, qrels)
    print(f"{name:8s}", "  ".join(f"{m} {v:.3f}" for m, v in scores.items()))

keyword  nDCG@10 0.688  R@100 0.931  MAP 0.647  MRR@10 0.652
dense    nDCG@10 0.645  R@100 0.925  MAP 0.603  MRR@10 0.605
hybrid   nDCG@10 0.713  R@100 0.964  MAP 0.672  MRR@10 0.678


Hybrid beats both halves: the claims share exact terms with their abstracts (gene names, drugs)
that the small embedding model blurs, while the embeddings find paraphrases that share no word.

## Where they differ

Per-query scores show which queries each run wins.

In [4]:
dense = ir.evaluate(runs["dense"], qrels, ["nDCG@10"], per_query=True)["nDCG@10"]
hybrid = ir.evaluate(runs["hybrid"], qrels, ["nDCG@10"], per_query=True)["nDCG@10"]
gains = sorted(((hybrid[q] - dense[q], q) for q in dense), reverse=True)
better = sum(g > 0 for g, _ in gains)
worse = sum(g < 0 for g, _ in gains)
print(f"hybrid better on {better} queries, worse on {worse}, the same on {len(gains) - better - worse}")
gain, q = gains[0]
print(f"largest gain (+{gain:.2f}): {queries[q]!r}")

hybrid better on 81 queries, worse on 30, the same on 189
largest gain (+1.00): 'Crossover hot spots are not found within gene promoters in Saccharomyces cerevisiae.'


## A run file

TREC's format, which trec_eval, CLEF and TREC all read. `read_run` loads one back, from any system.

In [5]:
ir.write_run(runs["hybrid"], "hybrid.run", tag="epzero-hybrid")
print(open("hybrid.run").read().splitlines()[0])
assert ir.evaluate(ir.read_run("hybrid.run"), qrels) == ir.evaluate(runs["hybrid"], qrels)

1 Q0 803312 1 0.025098 epzero-hybrid


## A changing collection

Documents can be removed, replaced and added by id; `health()` on the index says when its calibration
or codes need refreshing.

In [6]:
removed = collection.remove(doc_ids[:500])
collection.add(doc_ids[:500], doc_vectors[:500], texts=texts[:500])
print(removed, "removed and added back;", len(collection), "documents")
print(collection.index.health()["advice"])
collection.save("scifact-collection")
again = ir.Collection.load("scifact-collection")
print(ir.evaluate(again.retrieve(query_ids, vectors=query_vectors, k=100), qrels)["nDCG@10"])

500 removed and added back; 5183 documents
[]


0.6450816521455776
